In [1]:
import sys
import pandas as pd
import pickle
import importlib

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../')

# Add path to the other project src to handle imports with dashes in folder names
sys.path.insert(0, '../../../../../probabilistic_suffix_prediction_U-ED-LSTM/src')

In [2]:
# log as csv
event_log_path = '../../../../data/data/helpdesk.csv'
event_log_df = pd.read_csv(event_log_path)

In [3]:
# import petri net, 
petri_net_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/petri_net/helpdesk.pkl'
with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)

# split csv data
train_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_train.csv'
helpdesk_train_df = pd.read_csv(train_csv_path)

val_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_val.csv'
helpdesk_val_df = pd.read_csv(val_csv_path)

test_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_test.csv'
helpdesk_test_df = pd.read_csv(test_csv_path)

# get case ids
unique_list_train = helpdesk_train_df["CaseID"].dropna().unique().tolist()
unique_list_val = helpdesk_val_df["CaseID"].dropna().unique().tolist()

case_ids = list(dict.fromkeys(unique_list_train + unique_list_val))

In [4]:
import decision_mining.decision_discovery_custom
importlib.reload(decision_mining.decision_discovery_custom)
from decision_mining.decision_discovery_custom import DecisionDiscovery, ModelConfig, ModelType, GuardPolicy, GuardMode

dd = DecisionDiscovery(petri_net=(net, im, fm),
                       event_log_df=event_log_df,
                       case_ids=case_ids,
                       case_id_key='CaseID',
                       activity_key='Activity',
                       time_key='CompleteTimestamp')

dynamic_attributes = ['Resource']
static_attributes = ['VariantIndex', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup']

dd = DecisionDiscovery(petri_net=(net, im, fm),
                       event_log_df=event_log_df,
                       case_ids=case_ids,
                       case_id_key='CaseID',
                       activity_key='Activity',
                       time_key='CompleteTimestamp')

result = dd.mine_decision_models(attributes=dynamic_attributes,   # or your one-hot columns
                                 trace_attributes=static_attributes,
                                 model_cfg= ModelConfig(model_type=ModelType.AUTO, train_surrogate_tree=True),
                                 guard_policy= GuardPolicy(mode=GuardMode.THRESHOLD, threshold=0.15))


/home/PSPLab/.local/share/virtualenvs/probabilistic_suffix_prediction_U-ED-LSTM-939Sw3sq/lib/python3.12/site-packages/pm4py/visualization/decisiontree/__init__.py:28: UserWarning: The decisiontree visualizer will be removed in a future release (use Scikit Learn instead).
  warnings.warn(
/home/PSPLab/.local/share/virtualenvs/probabilistic_suffix_prediction_U-ED-LSTM-939Sw3sq/lib/python3.12/site-packages/pm4py/algo/decision_mining/__init__.py:30: UserWarning: The decision_mining package will be removed in a future release.
  warnings.warn(


{'p_20': DecisionPointSpec(place_name='p_20', pattern=<DecisionPattern.LOOP_OUTER: 'loop_outer'>, outgoing_transitions=[(skip_25, None), (2edf5665-8bfd-4f21-8b47-073879627eb2, 'Resolve ticket')], outgoing_branch_labels=['tau::skip_25', 'Resolve ticket'], visible_labels=['Resolve ticket'], tau_labels=['tau::skip_25'], loop_scc_id=1, loop_continue_branches=['tau::skip_25', 'Resolve ticket'], loop_exit_branches=[]), 'p_36': DecisionPointSpec(place_name='p_36', pattern=<DecisionPattern.LOOP_OUTER: 'loop_outer'>, outgoing_transitions=[(skip_40, None), (f63795ba-5a65-4e54-ab98-6f1af4d6f9ff, 'DUPLICATE')], outgoing_branch_labels=['tau::skip_40', 'DUPLICATE'], visible_labels=['DUPLICATE'], tau_labels=['tau::skip_40'], loop_scc_id=1, loop_continue_branches=['tau::skip_40', 'DUPLICATE'], loop_exit_branches=[]), 'p_12': DecisionPointSpec(place_name='p_12', pattern=<DecisionPattern.LOOP_OUTER: 'loop_outer'>, outgoing_transitions=[(skip_16, None), (e34a9610-c653-4413-ae8d-fc9dc3d43dca, 'Take in cha

In [5]:
# During decoding, at place p_20, given the current known assignment A (feature dict):
m = result.models["p_12"]
# allowed_branches = m.allowed_branches(A)
# allowed_activities = m.allowed_next_activities(A)  # directly usable as next-event mask

# print("branches:", allowed_branches)
# print("activities:", allowed_activities)

# If you want readable rules (surrogate):
print(m.surrogate_guard_by_branch)   # DNF-ish guards per branch (probability-based extraction)

AttributeError: 'NoneType' object has no attribute 'models'